# fhir4ds.cql

Clinical Quality Language (CQL) to SQL translation. This notebook demonstrates how CQL expressions are translated into highly optimized DuckDB SQL for population-scale analytics using the Zero-ETL architecture.

In [ ]:
# Install the unified package with measure evaluation support
%pip install "fhir4ds-v2[measures]"

In [ ]:
import json
import fhir4ds
from fhir4ds.cql import CQLToSQLTranslator
from fhir4ds.cql.parser import parse_cql
from fhir4ds.sources import ExistingTableSource

# Create connection
con = fhir4ds.create_connection()

# Set up a standard DuckDB table with FHIR data
con.execute("""
    CREATE TABLE my_fhir_table (
        id VARCHAR, 
        resourceType VARCHAR, 
        resource JSON, 
        patient_ref VARCHAR
    )
""")

resources = [
    ("p1", "Patient", '{"resourceType": "Patient", "id": "p1", "active": true}', "p1"),
    ("e1", "Encounter", '{"resourceType": "Encounter", "id": "e1", "status": "finished", "subject": {"reference": "Patient/p1"}}', "p1"),
]
for row in resources:
    con.execute("INSERT INTO my_fhir_table VALUES (?, ?, ?, ?)", row)

# Mount the existing table as the FHIR4DS 'resources' view
fhir4ds.attach(con, ExistingTableSource("my_fhir_table"))

### Parse and Translate
Translate CQL logic into inspectable SQL fragments. FHIR4DS supports complex joins and clinical logic natively in the database.

In [ ]:
cql_text = """
library Test version '1.0'
using FHIR version '4.0.1'
context Patient

define "Has Encounter":
    exists ([Encounter] E where E.status = 'finished')
"""

library = parse_cql(cql_text)
translator = CQLToSQLTranslator(connection=con)
definitions = translator.translate_library(library)

print(f"SQL for 'Has Encounter':\n{definitions['Has Encounter'].to_sql()}")

### Population SQL
The `translate_library_to_population_sql` method generates a single query that evaluates all definitions for the entire patient population in one scan.

In [ ]:
pop_sql = translator.translate_library_to_population_sql(
    library,
    output_columns={
        "has_encounter": "Has Encounter"
    }
)
con.execute(pop_sql).df()